# 06. AoU replication sample sizes and demographics

This notebook creates sample-size and demographic summaries for the **proteomics** and **clinical chemistry/biometric** AoU disease-association analyses.

It uses only project-owned tables already written by notebooks 01-04; it does **not** query BigQuery.

The principal demographic table describes unique participants who contributed to at least one successfully fitted primary AoU replication target within each modality. Because individual disease-feature models can have different missingness and case-control counts, the notebook also writes exact per-model sample-size ranges by disease domain.

Primary diversity reporting uses the `ancestry_pred` genetic-ancestry label from the same PC table used for model adjustment. Self-reported race and ethnicity are summarized separately for proteomics when those columns are present, but are not substituted for genetic ancestry.

In [ ]:
from pathlib import Path
import os
import sys


def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

protein_matrix_file = cfg.PROTEOMICS_DIR / "proteomics_analysis_matrix.tsv.gz"
chemistry_matrix_file = cfg.CHEMISTRY_DIR / "chemistry_analysis_matrix.tsv.gz"
protein_map_file = cfg.PROTEOMICS_DIR / "proteomics_feature_column_map.tsv"
chemistry_map_file = cfg.CHEMISTRY_DIR / "chemistry_feature_column_map.tsv"
primary_results_file = cfg.RESULTS_DIR / "combined_target_replication_results_primary.tsv"
pcs_file = cfg.SHARED_DIR / "aou_genetic_ancestry_pcs.tsv"

required = [
    protein_matrix_file,
    chemistry_matrix_file,
    protein_map_file,
    chemistry_map_file,
    primary_results_file,
    pcs_file,
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files. Run notebooks 01-04 first. Missing:\n  - "
        + "\n  - ".join(str(path) for path in missing)
    )

protein_matrix = pd.read_csv(
    protein_matrix_file, sep="\t", compression="gzip", low_memory=False
)
chemistry_matrix = pd.read_csv(
    chemistry_matrix_file, sep="\t", compression="gzip", low_memory=False
)
protein_map = pd.read_csv(protein_map_file, sep="\t", dtype="string")
chemistry_map = pd.read_csv(chemistry_map_file, sep="\t", dtype="string")
primary = pd.read_csv(primary_results_file, sep="\t", low_memory=False)
pcs = pd.read_csv(pcs_file, sep="\t", low_memory=False)

print("Proteomics matrix:", protein_matrix.shape)
print("Chemistry matrix:", chemistry_matrix.shape)
print("Primary result rows:", len(primary))
print("PC table:", pcs.shape)
print("Primary result columns:")
print(primary.columns.tolist())

## Existing data products

The upstream notebooks already write the following relevant files:

- `data/shared/aou_genetic_ancestry_pcs.tsv`: `person_id`, PC1-PC8 and, when available, `ancestry_pred`.
- `data/proteomics/proteomics_person_covariates.tsv`: birth fields, sex at birth, gender, self-reported race and ethnicity for the Olink participant set.
- `data/proteomics/proteomics_analysis_matrix.tsv.gz`: proteomic outcomes merged with age, sex, ancestry PCs, predicted ancestry and disease flags.
- `data/chemistries/chemistry_person_covariates.tsv`: age at selected measurements and sex at birth.
- `data/chemistries/chemistry_analysis_matrix.tsv.gz`: harmonized chemistry outcomes merged with age, sex, ancestry PCs, predicted ancestry and disease flags.
- `results/combined_target_replication_results_primary.tsv`: the primary model status and exact case/control N for every replication target.

The missing product was a single demographic summary across the actual primary replication samples; the following cells generate it.

In [ ]:
ANCESTRY_MAP = {
    "AFR": "African ancestry",
    "AFRICAN": "African ancestry",
    "EUR": "European ancestry",
    "EUROPEAN": "European ancestry",
    "EAS": "East Asian ancestry",
    "EAST ASIAN": "East Asian ancestry",
    "SAS": "South Asian ancestry",
    "SOUTH ASIAN": "South Asian ancestry",
    "AMR": "Admixed American ancestry",
    "ADMIXED AMERICAN": "Admixed American ancestry",
    "MID": "Middle Eastern ancestry",
    "MIDDLE EASTERN": "Middle Eastern ancestry",
}

ANCESTRY_ORDER = [
    "African ancestry",
    "Admixed American ancestry",
    "East Asian ancestry",
    "European ancestry",
    "Middle Eastern ancestry",
    "South Asian ancestry",
    "Other or unclassified ancestry",
]


def normalize_ancestry(value):
    if pd.isna(value):
        return "Other or unclassified ancestry"
    text = str(value).strip()
    if not text:
        return "Other or unclassified ancestry"
    upper = text.upper().replace("_", " ").replace("-", " ")
    upper = " ".join(upper.split())
    if upper in ANCESTRY_MAP:
        return ANCESTRY_MAP[upper]
    for key, label in ANCESTRY_MAP.items():
        if key in upper:
            return label
    return text


def truthy(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0).eq(1)
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y", "t"])
    )


def load_safe_feature_map(table):
    needed = {"feature_key", "outcome_col", "available_for_model"}
    absent = needed - set(table.columns)
    if absent:
        raise ValueError(f"Feature map lacks columns: {sorted(absent)}")

    usable = table.loc[
        truthy(table["available_for_model"])
        & table["feature_key"].notna()
        & table["outcome_col"].notna(),
        ["feature_key", "outcome_col"],
    ].copy()
    usable["feature_key"] = usable["feature_key"].str.strip()
    usable["outcome_col"] = usable["outcome_col"].str.strip()
    usable = usable.drop_duplicates()

    conflicts = usable.groupby("feature_key")["outcome_col"].nunique()
    conflicts = conflicts[conflicts > 1]
    if not conflicts.empty:
        raise ValueError(
            "At least one feature key maps to multiple matrix columns: "
            + ", ".join(conflicts.index[:10])
        )
    return dict(zip(usable["feature_key"], usable["outcome_col"]))


protein_feature_map = load_safe_feature_map(protein_map)
chemistry_feature_map = load_safe_feature_map(chemistry_map)


def ensure_ancestry_columns(matrix):
    matrix = matrix.copy()
    matrix["person_id"] = pd.to_numeric(
        matrix["person_id"], errors="raise"
    ).astype("int64")

    if "ancestry_pred" not in matrix.columns:
        pc_keep = ["person_id", "ancestry_pred", *cfg.PC_COLUMNS]
        pc_keep = [column for column in pc_keep if column in pcs.columns]
        matrix = matrix.merge(
            pcs[pc_keep].drop_duplicates("person_id"),
            on="person_id",
            how="left",
            validate="one_to_one",
        )

    if "has_complete_pc8" not in matrix.columns:
        matrix["has_complete_pc8"] = matrix[cfg.PC_COLUMNS].notna().all(axis=1)

    matrix["genetic_ancestry"] = matrix["ancestry_pred"].map(normalize_ancestry)
    return matrix


protein_matrix = ensure_ancestry_columns(protein_matrix)
chemistry_matrix = ensure_ancestry_columns(chemistry_matrix)

print("Proteomics ancestry labels:")
display(protein_matrix["genetic_ancestry"].value_counts(dropna=False))
print("Chemistry ancestry labels:")
display(chemistry_matrix["genetic_ancestry"].value_counts(dropna=False))

In [ ]:
def get_status_column(table):
    for column in [
        "discovery_concordant_status",
        "status",
        "fit_status",
        "model_status",
    ]:
        if column in table.columns:
            return column
    raise ValueError("No recognizable model-status column was found")


def resolve_control_column(matrix, domain):
    preferred = [
        f"{domain}_domain_control_candidate",
        f"{domain}_control_candidate",
        f"{domain}_control",
    ]
    for column in preferred:
        if column in matrix.columns:
            return column

    candidates = [
        column
        for column in matrix.columns
        if column.startswith(f"{domain}_")
        and "control" in column.lower()
        and "candidate" in column.lower()
    ]
    if len(candidates) == 1:
        return candidates[0]
    raise KeyError(
        f"Could not resolve a control column for {domain}. Candidates: {candidates}"
    )


def successful_primary_pairs(primary_table, omic_type, feature_map, matrix):
    status_col = get_status_column(primary_table)
    subset = primary_table.loc[
        primary_table["omic_type"].astype(str).eq(omic_type)
        & primary_table[status_col].astype(str).eq("ok")
    ].copy()

    if "outcome_col" not in subset.columns:
        subset["outcome_col"] = subset["feature_key"].map(feature_map)
    else:
        missing_outcome = subset["outcome_col"].isna()
        subset.loc[missing_outcome, "outcome_col"] = (
            subset.loc[missing_outcome, "feature_key"].map(feature_map)
        )

    subset = subset.loc[
        subset["analysis_domain"].notna()
        & subset["outcome_col"].notna()
        & subset["outcome_col"].isin(matrix.columns)
    ].copy()

    return (
        subset[["analysis_domain", "feature_key", "outcome_col"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )


def build_analysis_masks(matrix, age_col, pairs):
    base = (
        truthy(matrix["has_complete_pc8"])
        & pd.to_numeric(matrix[age_col], errors="coerce").notna()
        & matrix["sex_adjustment"].notna()
    )

    modeled_outcomes = sorted(set(pairs["outcome_col"]))
    any_modeled_outcome = matrix[modeled_outcomes].notna().any(axis=1)
    modality_pool = base & any_modeled_outcome

    any_primary_model = pd.Series(False, index=matrix.index)
    domain_masks = {}

    for domain, domain_pairs in pairs.groupby("analysis_domain"):
        case_col = f"{domain}_case"
        if case_col not in matrix.columns:
            raise KeyError(f"Missing case column: {case_col}")
        control_col = resolve_control_column(matrix, domain)

        domain_membership = truthy(matrix[case_col]) | truthy(matrix[control_col])
        domain_outcomes = sorted(set(domain_pairs["outcome_col"]))
        has_domain_outcome = matrix[domain_outcomes].notna().any(axis=1)
        domain_mask = base & domain_membership & has_domain_outcome

        domain_masks[domain] = domain_mask
        any_primary_model = any_primary_model | domain_mask

    return {
        "modality_pool": modality_pool,
        "any_primary_model": any_primary_model,
        "domain_masks": domain_masks,
    }


protein_pairs = successful_primary_pairs(
    primary, "proteomics", protein_feature_map, protein_matrix
)
chemistry_pairs = successful_primary_pairs(
    primary, "chemistries", chemistry_feature_map, chemistry_matrix
)

protein_masks = build_analysis_masks(
    protein_matrix, "age_at_assay", protein_pairs
)
chemistry_masks = build_analysis_masks(
    chemistry_matrix, "age_at_measurement", chemistry_pairs
)

print("Successful unique primary protein domain-feature pairs:", len(protein_pairs))
print("Successful unique primary chemistry domain-feature pairs:", len(chemistry_pairs))
print()
print("Proteomics PC/covariate-complete pool with >=1 modeled outcome:",
      f"{protein_masks['modality_pool'].sum():,}")
print("Proteomics contributors to >=1 successful primary model:",
      f"{protein_masks['any_primary_model'].sum():,}")
print("Chemistry PC/covariate-complete pool with >=1 modeled outcome:",
      f"{chemistry_masks['modality_pool'].sum():,}")
print("Chemistry contributors to >=1 successful primary model:",
      f"{chemistry_masks['any_primary_model'].sum():,}")

In [ ]:
def n_pct(n, denominator):
    if denominator == 0:
        return "0 (0.0%)"
    return f"{int(n):,} ({100.0 * n / denominator:.1f}%)"


def summarize_demographics(matrix, mask, age_col, modality, scope):
    data = matrix.loc[mask].copy()
    n = len(data)
    age = pd.to_numeric(data[age_col], errors="coerce").dropna()

    rows = [
        {
            "scope": scope,
            "modality": modality,
            "characteristic": "Unique participants, N",
            "value": f"{n:,}",
            "n": n,
            "percent": np.nan,
        },
        {
            "scope": scope,
            "modality": modality,
            "characteristic": "Age, years, mean (SD)",
            "value": f"{age.mean():.1f} ({age.std(ddof=1):.1f})",
            "n": len(age),
            "percent": np.nan,
        },
        {
            "scope": scope,
            "modality": modality,
            "characteristic": "Age, years, median [IQR]",
            "value": (
                f"{age.median():.1f} "
                f"[{age.quantile(0.25):.1f}-{age.quantile(0.75):.1f}]"
            ),
            "n": len(age),
            "percent": np.nan,
        },
    ]

    sex = data["sex_adjustment"].fillna("Other/Unknown").astype(str)
    for label in ["Female", "Male", "Other/Unknown"]:
        count = int(sex.eq(label).sum())
        rows.append(
            {
                "scope": scope,
                "modality": modality,
                "characteristic": f"{label} sex, n (%)",
                "value": n_pct(count, n),
                "n": count,
                "percent": 100.0 * count / n if n else np.nan,
            }
        )

    ancestry = data["genetic_ancestry"].fillna(
        "Other or unclassified ancestry"
    )
    observed_order = [
        label for label in ANCESTRY_ORDER if label in set(ancestry)
    ]
    extras = sorted(set(ancestry) - set(observed_order))

    for label in [*observed_order, *extras]:
        count = int(ancestry.eq(label).sum())
        rows.append(
            {
                "scope": scope,
                "modality": modality,
                "characteristic": f"{label}, n (%)",
                "value": n_pct(count, n),
                "n": count,
                "percent": 100.0 * count / n if n else np.nan,
            }
        )

    return pd.DataFrame(rows)


summaries = []
for modality, matrix, masks, age_col in [
    ("Proteomics", protein_matrix, protein_masks, "age_at_assay"),
    ("Chemistries", chemistry_matrix, chemistry_masks, "age_at_measurement"),
]:
    summaries.append(
        summarize_demographics(
            matrix,
            masks["modality_pool"],
            age_col,
            modality,
            "PC/covariate-complete modality pool",
        )
    )
    summaries.append(
        summarize_demographics(
            matrix,
            masks["any_primary_model"],
            age_col,
            modality,
            "Contributed to >=1 successful primary model",
        )
    )

demographics_long = pd.concat(summaries, ignore_index=True)

main_scope = "Contributed to >=1 successful primary model"
demographics_main = demographics_long.loc[
    demographics_long["scope"].eq(main_scope)
].copy()
demographics_wide = (
    demographics_main.pivot(
        index="characteristic", columns="modality", values="value"
    )
    .reset_index()
)

# Preserve a manuscript-friendly row order.
row_order = [
    "Unique participants, N",
    "Age, years, mean (SD)",
    "Age, years, median [IQR]",
    "Female sex, n (%)",
    "Male sex, n (%)",
    "Other/Unknown sex, n (%)",
] + [f"{label}, n (%)" for label in ANCESTRY_ORDER]
rank = {label: index for index, label in enumerate(row_order)}
demographics_wide["_order"] = demographics_wide["characteristic"].map(rank).fillna(999)
demographics_wide = demographics_wide.sort_values(["_order", "characteristic"]).drop(columns="_order")

print("Manuscript-ready AoU demographic table")
display(demographics_wide)

In [ ]:
def exact_model_n_columns(table):
    case_candidates = [
        "discovery_concordant_n_cases",
        "n_cases",
    ]
    control_candidates = [
        "discovery_concordant_n_controls",
        "n_controls",
    ]
    case_col = next((c for c in case_candidates if c in table.columns), None)
    control_col = next((c for c in control_candidates if c in table.columns), None)
    if case_col is None or control_col is None:
        raise ValueError(
            "Primary results lack recognizable exact case/control N columns"
        )
    return case_col, control_col


case_n_col, control_n_col = exact_model_n_columns(primary)
status_col = get_status_column(primary)
modelled = primary.loc[primary[status_col].astype(str).eq("ok")].copy()
modelled[case_n_col] = pd.to_numeric(modelled[case_n_col], errors="coerce")
modelled[control_n_col] = pd.to_numeric(modelled[control_n_col], errors="coerce")

model_sample_ranges = (
    modelled.groupby(["omic_type", "analysis_domain"], as_index=False)
    .agg(
        n_successful_targets=("analysis_target_id", "nunique"),
        n_cases_min=(case_n_col, "min"),
        n_cases_median=(case_n_col, "median"),
        n_cases_max=(case_n_col, "max"),
        n_controls_min=(control_n_col, "min"),
        n_controls_median=(control_n_col, "median"),
        n_controls_max=(control_n_col, "max"),
    )
)

for column in [
    "n_cases_min", "n_cases_max", "n_controls_min", "n_controls_max"
]:
    model_sample_ranges[column] = model_sample_ranges[column].round().astype("Int64")
for column in ["n_cases_median", "n_controls_median"]:
    model_sample_ranges[column] = model_sample_ranges[column].round(1)

print("Exact per-target sample-size ranges from the fitted primary models")
display(model_sample_ranges)

In [ ]:
# Domain-level demographic summaries use the union of participants contributing
# to at least one successfully fitted target in that disease domain.
def summarize_domains(matrix, masks, age_col, modality):
    rows = []
    for domain, mask in masks["domain_masks"].items():
        data = matrix.loc[mask].copy()
        n = len(data)
        age = pd.to_numeric(data[age_col], errors="coerce")
        ancestry_counts = (
            data["genetic_ancestry"]
            .fillna("Other or unclassified ancestry")
            .value_counts()
        )
        base = {
            "modality": modality,
            "analysis_domain": domain,
            "n_unique_participants": n,
            "age_mean": age.mean(),
            "age_sd": age.std(ddof=1),
            "female_n": int(data["sex_adjustment"].eq("Female").sum()),
            "male_n": int(data["sex_adjustment"].eq("Male").sum()),
        }
        for ancestry_label, count in ancestry_counts.items():
            rows.append(
                {
                    **base,
                    "genetic_ancestry": ancestry_label,
                    "ancestry_n": int(count),
                    "ancestry_percent": 100.0 * count / n if n else np.nan,
                }
            )
    return pd.DataFrame(rows)


domain_demographics = pd.concat(
    [
        summarize_domains(
            protein_matrix, protein_masks, "age_at_assay", "Proteomics"
        ),
        summarize_domains(
            chemistry_matrix, chemistry_masks, "age_at_measurement", "Chemistries"
        ),
    ],
    ignore_index=True,
)

display(domain_demographics.head(30))

In [ ]:
# Modality overlap is useful when describing whether the two analyses represent
# the same participants or partially overlapping AoU subsets.
protein_ids = set(
    protein_matrix.loc[
        protein_masks["any_primary_model"], "person_id"
    ].astype("int64")
)
chemistry_ids = set(
    chemistry_matrix.loc[
        chemistry_masks["any_primary_model"], "person_id"
    ].astype("int64")
)

overlap_summary = pd.DataFrame(
    [
        {"measure": "Proteomics unique participants", "n": len(protein_ids)},
        {"measure": "Chemistry unique participants", "n": len(chemistry_ids)},
        {"measure": "Participants in both modalities", "n": len(protein_ids & chemistry_ids)},
        {"measure": "Participants in either modality", "n": len(protein_ids | chemistry_ids)},
    ]
)
display(overlap_summary)

# Self-reported race and ethnicity were explicitly queried in notebook 02 for
# proteomics. They were not queried in notebook 03 for the chemistry cohort, so
# this optional table is deliberately proteomics-only.
race_ethnicity_rows = []
protein_used = protein_matrix.loc[protein_masks["any_primary_model"]].copy()
for variable in ["race", "ethnicity"]:
    if variable in protein_used.columns:
        counts = protein_used[variable].fillna("Missing/unknown").astype(str).value_counts()
        for category, count in counts.items():
            race_ethnicity_rows.append(
                {
                    "variable": variable,
                    "category": category,
                    "n": int(count),
                    "percent": 100.0 * count / len(protein_used),
                }
            )
proteomics_race_ethnicity = pd.DataFrame(race_ethnicity_rows)
if not proteomics_race_ethnicity.empty:
    display(proteomics_race_ethnicity)

In [ ]:
output_dir = cfg.RESULTS_DIR / "demographics"
output_dir.mkdir(parents=True, exist_ok=True)

demographics_long.to_csv(
    output_dir / "aou_replication_demographics_all_scopes.tsv",
    sep="\t",
    index=False,
)
demographics_wide.to_csv(
    output_dir / "aou_replication_demographics_manuscript_ready.tsv",
    sep="\t",
    index=False,
)
model_sample_ranges.to_csv(
    output_dir / "aou_replication_exact_model_sample_size_ranges.tsv",
    sep="\t",
    index=False,
)
domain_demographics.to_csv(
    output_dir / "aou_replication_demographics_by_domain_and_modality.tsv",
    sep="\t",
    index=False,
)
overlap_summary.to_csv(
    output_dir / "aou_replication_modality_overlap.tsv",
    sep="\t",
    index=False,
)
if not proteomics_race_ethnicity.empty:
    proteomics_race_ethnicity.to_csv(
        output_dir / "aou_proteomics_self_reported_race_ethnicity.tsv",
        sep="\t",
        index=False,
    )

print("Saved demographic outputs to:", output_dir)
for path in sorted(output_dir.glob("aou_*.tsv")):
    print(" -", path.name)

In [ ]:
def ancestry_phrase(matrix, mask, max_categories=6):
    data = matrix.loc[mask]
    n = len(data)
    counts = data["genetic_ancestry"].fillna(
        "Other or unclassified ancestry"
    ).value_counts()
    parts = []
    for label, count in counts.head(max_categories).items():
        parts.append(f"{label} {count:,} ({100.0 * count / n:.1f}%)")
    return "; ".join(parts)

n_protein = int(protein_masks["any_primary_model"].sum())
n_chemistry = int(chemistry_masks["any_primary_model"].sum())

print("AoU analytic-cohort summary:\n")
print(
    f"The AoU disease-association analyses included {n_protein:,} unique participants "
    f"contributing to at least one successfully fitted proteomic target and "
    f"{n_chemistry:,} contributing to at least one successfully fitted "
    "clinical chemistry or biometric target. The ancestry-adjusted analytic "
    "samples were genetically diverse. Proteomics: "
    + ancestry_phrase(protein_matrix, protein_masks["any_primary_model"])
    + ". Chemistries: "
    + ancestry_phrase(chemistry_matrix, chemistry_masks["any_primary_model"])
    + ". Individual disease-feature models used target-specific subsets; exact "
      "case and control sample-size ranges are reported in the accompanying table."
)

In [ ]:
# =============================================================================
# Disease-specific case/control sample sizes for manuscript reporting
# =============================================================================
#
# Produces:
#   1. A manuscript-ready wide table with case/control N or N ranges
#      for each disease and modality.
#   2. A long-form audit table retaining the underlying minimum, median
#      and maximum values.
#   3. A caption describing the case and control definitions.
#
# Ranges arise because outcome availability and complete-covariate
# requirements can differ across individual biomarker models.

from IPython.display import display, Markdown

DOMAIN_LABELS = {
    "AD_dementia": "AD/dementia",
    "Parkinsons": "PD",
    "ALS_MND": "ALS",
    "ALS": "ALS",
    "CAD_coronary": "CAD/coronary disease",
    "CVD_composite": "Composite CVD",
    "Heart_failure": "Heart failure",
    "T2D": "T2D",
}

DOMAIN_ORDER = [
    "AD_dementia",
    "Parkinsons",
    "ALS",
    "CAD_coronary",
    "CVD_composite",
    "Heart_failure",
    "T2D",
]

MODALITY_LABELS = {
    "chemistries": "Chemistries",
    "proteomics": "Proteomics",
}


def format_n_range(minimum, maximum):
    """Format an exact N or an en-dash-separated N range."""
    if pd.isna(minimum) or pd.isna(maximum):
        return "-"

    minimum = int(round(float(minimum)))
    maximum = int(round(float(maximum)))

    if minimum == maximum:
        return f"{minimum:,}"

    return f"{minimum:,} to {maximum:,}"


# Resolve the exact fitted-model case/control columns using the helper
# already defined earlier in this notebook.
case_n_col, control_n_col = exact_model_n_columns(primary)
status_col = get_status_column(primary)

sample_rows = primary.loc[
    primary[status_col].astype(str).eq("ok")
    & primary["analysis_domain"].notna()
    & primary["omic_type"].astype(str).isin(MODALITY_LABELS)
].copy()

if sample_rows.empty:
    raise ValueError(
        "No successfully fitted primary replication rows were found."
    )

# Normalize ALS naming so that the table uses the manuscript terminology.
sample_rows["analysis_domain"] = (
    sample_rows["analysis_domain"]
    .astype(str)
    .replace({"ALS_MND": "ALS"})
)

sample_rows[case_n_col] = pd.to_numeric(
    sample_rows[case_n_col], errors="coerce"
)
sample_rows[control_n_col] = pd.to_numeric(
    sample_rows[control_n_col], errors="coerce"
)

sample_rows = sample_rows.loc[
    sample_rows[case_n_col].notna()
    & sample_rows[control_n_col].notna()
].copy()

# Prefer the actual AoU outcome column as the unique biomarker/test identifier.
# Fall back to the harmonized feature key or analysis-target identifier if needed.
biomarker_id_col = next(
    (
        column
        for column in ["outcome_col", "feature_key", "analysis_target_id"]
        if column in sample_rows.columns
    ),
    None,
)

if biomarker_id_col is None:
    raise ValueError(
        "Could not identify a biomarker/test identifier column in the "
        "primary replication results."
    )

# Deduplicate repeated discovery records linked to the same fitted AoU outcome.
# Different Ns for the same outcome are retained if the fitted estimand or
# complete-case population genuinely differed.
unique_model_samples = sample_rows[
    [
        "omic_type",
        "analysis_domain",
        biomarker_id_col,
        case_n_col,
        control_n_col,
    ]
].drop_duplicates()

case_control_long = (
    unique_model_samples
    .groupby(["analysis_domain", "omic_type"], as_index=False)
    .agg(
        n_biomarkers=(biomarker_id_col, "nunique"),
        cases_min=(case_n_col, "min"),
        cases_median=(case_n_col, "median"),
        cases_max=(case_n_col, "max"),
        controls_min=(control_n_col, "min"),
        controls_median=(control_n_col, "median"),
        controls_max=(control_n_col, "max"),
    )
)

for column in [
    "n_biomarkers",
    "cases_min",
    "cases_max",
    "controls_min",
    "controls_max",
]:
    case_control_long[column] = (
        case_control_long[column].round().astype("Int64")
    )

for column in ["cases_median", "controls_median"]:
    case_control_long[column] = case_control_long[column].round(1)

case_control_long["disease"] = case_control_long["analysis_domain"].map(
    DOMAIN_LABELS
)
case_control_long["modality"] = case_control_long["omic_type"].map(
    MODALITY_LABELS
)
case_control_long["cases_n"] = case_control_long.apply(
    lambda row: format_n_range(row["cases_min"], row["cases_max"]),
    axis=1,
)
case_control_long["controls_n"] = case_control_long.apply(
    lambda row: format_n_range(row["controls_min"], row["controls_max"]),
    axis=1,
)

# Construct the wide, manuscript-ready table.
manuscript_rows = []

for domain in DOMAIN_ORDER:
    disease_label = DOMAIN_LABELS[domain]
    row = {"Disease": disease_label}

    for modality_key, modality_label in [
        ("chemistries", "Chemistries"),
        ("proteomics", "Proteomics"),
    ]:
        subset = case_control_long.loc[
            case_control_long["analysis_domain"].eq(domain)
            & case_control_long["omic_type"].eq(modality_key)
        ]

        if subset.empty:
            row[f"{modality_label} biomarkers, n"] = "—"
            row[f"{modality_label} cases, n"] = "—"
            row[f"{modality_label} controls, n"] = "—"
        else:
            result = subset.iloc[0]
            row[f"{modality_label} biomarkers, n"] = (
                f"{int(result['n_biomarkers']):,}"
            )
            row[f"{modality_label} cases, n"] = result["cases_n"]
            row[f"{modality_label} controls, n"] = result["controls_n"]

    manuscript_rows.append(row)

case_control_manuscript = pd.DataFrame(manuscript_rows)

# Drop disease rows that had no successfully fitted models in either modality.
case_control_manuscript = case_control_manuscript.loc[
    ~(
        case_control_manuscript["Chemistries biomarkers, n"].eq("—")
        & case_control_manuscript["Proteomics biomarkers, n"].eq("—")
    )
].reset_index(drop=True)

caption = (
    "Disease-specific sample sizes for All of Us external phenotypic "
    "validation. Cases were participants with qualifying electronic health "
    "record diagnoses from curated OMOP condition concept sets expanded "
    "through descendant concepts, after manual exclusion of non-diagnostic "
    "concepts. Primary controls were disease-domain-specific participants "
    "who were neither cases nor members of the broader exclusion concept "
    "set for that disease. The composite cardiovascular disease endpoint "
    "was defined as the union of CAD/coronary disease and heart failure. "
    "Values are shown as an exact N when all successfully fitted biomarker "
    "models used the same analytic sample, or as the minimum–maximum range "
    "across models when biomarker availability or complete-covariate "
    "requirements produced target-specific samples. No pre-diagnosis "
    "measurement restriction was imposed; these analyses evaluate "
    "cross-sectional differences between diagnosed cases and controls."
)

print("Manuscript-ready disease-specific case/control sample sizes")
display(case_control_manuscript)

display(Markdown("**Suggested table caption**"))
display(Markdown(caption))

# Save outputs.
output_dir = cfg.RESULTS_DIR / "demographics"
output_dir.mkdir(parents=True, exist_ok=True)

case_control_long.to_csv(
    output_dir / "aou_replication_case_control_n_by_disease_long.tsv",
    sep="\t",
    index=False,
)

case_control_manuscript.to_csv(
    output_dir
    / "aou_replication_case_control_n_by_disease_manuscript_ready.tsv",
    sep="\t",
    index=False,
)

case_control_manuscript.to_csv(
    output_dir
    / "aou_replication_case_control_n_by_disease_manuscript_ready.csv",
    index=False,
)

(
    output_dir / "aou_replication_case_control_table_caption.txt"
).write_text(caption + "\n", encoding="utf-8")

print("\nSaved:")
print(
    " -",
    output_dir
    / "aou_replication_case_control_n_by_disease_manuscript_ready.tsv",
)
print(
    " -",
    output_dir / "aou_replication_case_control_n_by_disease_long.tsv",
)
print(
    " -",
    output_dir / "aou_replication_case_control_table_caption.txt",
)